In [1]:
import logging

import pandas as pd
import torch
import torchtext

from models import *
from preprocessing import *

print("PyTorch Version : {}".format(torch.__version__))
print("Torch Text Version : {}".format(torchtext.__version__))

/home/sigma/projects/PycharmProjects/toxic-comment-detection/venv/lib/python3.11/site-packages/torchtext/data/utils.py:105: UserWarning: Spacy model "en" could not be loaded, trying "en_core_web_sm" instead
  warnings.warn(


PyTorch Version : 2.0.1+cu118
Torch Text Version : 0.15.2+cpu


In [2]:
%env WANDB_SILENT=True


logger = logging.getLogger("wandb")
logger.setLevel(logging.ERROR)

env: WANDB_SILENT=True


In [3]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda:0")

device

device(type='cuda', index=0)

In [4]:
# Load Datasets And Create Data Loaders

train_df = pd.read_csv("data/train.csv")
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
BATCH_SIZE = 256
# df_frac = 0.3
df_frac = 1.0
num_epochs = 50

## CNN (50 tokens - GlovE 300 - adam)

In [6]:
train_dl_50t_glove_300, val_dl_50t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:14, 1724.01it/s]
31915it [00:17, 1775.35it/s]


In [7]:
cnn = CNN(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=50,
).to(device)

cnn.train_model(
    train_dataloader=train_dl_50t_glove_300,
    validation_dataloader=val_dl_50t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN (50 tokens - Glove_300 - adam)",
    architecture="CNN",
)

Epoch 1:
{   'Training AUROC': 0.9587835669517517,
    'Training F1_score': 0.6245858073234558,
    'Training accuracy': 0.9763322472572327,
    'Training loss': 0.07209946920270313,
    'Training precision': 0.7444339990615845,
    'Training recall': 0.5379757881164551,
    'Validation AUROC': 0.9811472296714783,
    'Validation F1_score': 0.7361669540405273,
    'Validation accuracy': 0.9821714162826538,
    'Validation loss': 0.05158030317723751,
    'Validation precision': 0.8109995126724243,
    'Validation recall': 0.6739776134490967}
Epoch 2:
{   'Training AUROC': 0.9848505258560181,
    'Training F1_score': 0.7371102571487427,
    'Training accuracy': 0.9826186299324036,
    'Training loss': 0.04791243240341753,
    'Training precision': 0.8254754543304443,
    'Training recall': 0.6658342480659485,
    'Validation AUROC': 0.9836298227310181,
    'Validation F1_score': 0.7369416356086731,
    'Validation accuracy': 0.9825630784034729,
    'Validation loss': 0.04848878310620785,

## CNN (25 tokens - GlovE 300 - adam)

In [6]:
train_dl_25t_glove_300, val_dl_25t_glove_300 = get_dataloaders(
    target_convertor="glove_300",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:06, 1911.68it/s]
31915it [00:15, 2080.50it/s]


In [7]:
cnn = CNN(
    embed_dim=300,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=25,
).to(device)

cnn.train_model(
    train_dataloader=train_dl_25t_glove_300,
    validation_dataloader=val_dl_25t_glove_300,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN (25 tokens - Glove_300 - adam)",
    architecture="CNN",
)

Epoch 1:
{   'Training AUROC': 0.9474176168441772,
    'Training F1_score': 0.6069886088371277,
    'Training accuracy': 0.9762408137321472,
    'Training loss': 0.07788262605099736,
    'Training precision': 0.7728647589683533,
    'Training recall': 0.4997332990169525,
    'Validation AUROC': 0.973615288734436,
    'Validation F1_score': 0.6886587738990784,
    'Validation accuracy': 0.980531632900238,
    'Validation loss': 0.05779391586780548,
    'Validation precision': 0.8250950574874878,
    'Validation recall': 0.5909416675567627}
Epoch 2:
{   'Training AUROC': 0.9758875966072083,
    'Training F1_score': 0.7105407118797302,
    'Training accuracy': 0.9813600778579712,
    'Training loss': 0.05543645053265807,
    'Training precision': 0.8264786601066589,
    'Training recall': 0.6231285929679871,
    'Validation AUROC': 0.9765660762786865,
    'Validation F1_score': 0.6982178688049316,
    'Validation accuracy': 0.9812523126602173,
    'Validation loss': 0.055105574488639834,


## CNN (50 tokens - GlovE 100 - adam)

In [6]:
train_dl_50t_glove_100, val_dl_50t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=50,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:08, 1877.15it/s]
31915it [00:18, 1714.77it/s]


In [7]:
cnn = CNN(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=50,
).to(device)

cnn.train_model(
    train_dataloader=train_dl_50t_glove_100,
    validation_dataloader=val_dl_50t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN (50 tokens - Glove_100 - adam)",
    architecture="CNN",
)

Epoch 1:
{   'Training AUROC': 0.9229823350906372,
    'Training F1_score': 0.4444063901901245,
    'Training accuracy': 0.9703421592712402,
    'Training loss': 0.0950071919328703,
    'Training precision': 0.7091562151908875,
    'Training recall': 0.3235975205898285,
    'Validation AUROC': 0.9605196714401245,
    'Validation F1_score': 0.5966313481330872,
    'Validation accuracy': 0.9757376313209534,
    'Validation loss': 0.07262151789665222,
    'Validation precision': 0.7644048929214478,
    'Validation recall': 0.4892496168613434}
Epoch 2:
{   'Training AUROC': 0.9640941619873047,
    'Training F1_score': 0.5971438884735107,
    'Training accuracy': 0.9761337637901306,
    'Training loss': 0.06921825767728035,
    'Training precision': 0.7830761075019836,
    'Training recall': 0.4825645685195923,
    'Validation AUROC': 0.9666355848312378,
    'Validation F1_score': 0.5908465385437012,
    'Validation accuracy': 0.9767037630081177,
    'Validation loss': 0.06759135204553604,


## CNN (25 tokens - GlovE 100 - adam)

In [6]:
train_dl_25t_glove_100, val_dl_25t_glove_100 = get_dataloaders(
    target_convertor="glove_100",
    maximum_tokens=25,
    batch_size=BATCH_SIZE,
    dataset_fraction=df_frac,
)

127656it [01:05, 1951.53it/s]
31915it [00:14, 2180.51it/s]


In [7]:
cnn = CNN(
    embed_dim=100,
    conv_config={"num_channels": 50, "kernel_sizes": [1, 2, 3, 5]},
    optimizer_type="adam",
    learning_rate=0.001,
    maximum_tokens=25,
).to(device)

cnn.train_model(
    train_dataloader=train_dl_25t_glove_100,
    validation_dataloader=val_dl_25t_glove_100,
    num_of_epochs=num_epochs,
    device=device,
    name="CNN (25 tokens - Glove_100 - adam)",
    architecture="CNN",
)

Epoch 1:
{   'Training AUROC': 0.9142701625823975,
    'Training F1_score': 0.42945635318756104,
    'Training accuracy': 0.9699374437332153,
    'Training loss': 0.09806364629126503,
    'Training precision': 0.7017572522163391,
    'Training recall': 0.3094005584716797,
    'Validation AUROC': 0.944040834903717,
    'Validation F1_score': 0.5522215366363525,
    'Validation accuracy': 0.9737375378608704,
    'Validation loss': 0.08164691612124443,
    'Validation precision': 0.7486721277236938,
    'Validation recall': 0.4374382793903351}
Epoch 2:
{   'Training AUROC': 0.9527591466903687,
    'Training F1_score': 0.5652671456336975,
    'Training accuracy': 0.9752067923545837,
    'Training loss': 0.07531066005209644,
    'Training precision': 0.7877241373062134,
    'Training recall': 0.4407868981361389,
    'Validation AUROC': 0.9523226022720337,
    'Validation F1_score': 0.5731014609336853,
    'Validation accuracy': 0.9748707413673401,
    'Validation loss': 0.07602406620979309,